In [1365]:
import pandas as pd
import numpy as np
import string
import os
import re
import matplotlib.pyplot as plt
import seaborn as sns
from typing import Optional, Dict, List, Tuple
from modules.ruca_classification import classify_ruca_from_lat_lon
from modules.weighting_processing import main, parse_location_data, load_configurations
from modules.yaml_processing_helpers import load_mappings_from_yaml, map_industries, create_mapping, create_dummies_from_text
import reverse_geocoder as rg
import ses_demographic_data_build as ses
import Survey_Weighting_Module as p1_wg

In [3]:
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
%load_ext autoreload 
%autoreload 2

In [1222]:
configs = load_configurations()
mcs_2022 = os.path.join(configs.processed_data_drive_path, 'mcs_2022.csv')
mcs_2023 = os.path.join(configs.processed_data_drive_path, 'mcs_2023.csv')
mcs_2024 = os.path.join(configs.processed_data_drive_path, 'mcs_2024.csv')
mcs_2025 = os.path.join(configs.processed_data_drive_path, 'mcs_2025.csv')

mcs_2022 = pd.read_csv(mcs_2022)
mcs_2023 = pd.read_csv(mcs_2023)
mcs_2024 = pd.read_csv(mcs_2024)
mcs_2025 = pd.read_csv(mcs_2025)

In [5]:
cols = ['record_id', 'work_remote']

In [14]:
remote_map = {
    'always': 'Remote', 
    'almost always': 'Hybrid',
    'sometimes': 'Hybrid',
    'never': 'In-Person',
    'hybrid': 'Hybrid', 
    'remote': 'Remote', 
    'in_person': 'In-Person', 
    'never': 'In-Person',
    'frequent_hybrid': 'Hybrid', 
    'fully_remote': 'Remote', 
    'infrequent_hybrid': 'Hybrid',
    'multiple_worksites': 'Mobile'
}

In [15]:
stated_telework = pd.concat([mcs_2022[cols], mcs_2023[cols], mcs_2024[cols], mcs_2025[cols]])

In [16]:
stated_telework['work_remote_clean'] = stated_telework.work_remote.map(remote_map)

In [52]:
cols = ['record_id', 
        'hh_income', 
        'education', 
        'age', 
        'gender', 
        'race_ethnicity',
        'race_ethnicity_text',
        'disability',
        'work_status',
        'hh_size',
        'hh_children',
        'hh_vehicles']

In [55]:
mcs_2022 = mcs_2022.rename(columns = {'people_in_hh' :'hh_size'})
mcs_2022 = mcs_2022.rename(columns = {'children_in_hh' :'hh_children'})
mcs_2022 = mcs_2022.rename(columns = {'employment' :'work_status'})
mcs_2022 = mcs_2022.rename(columns = {'cars_in_hh' :'hh_vehicles'})

In [56]:
ses_demographic_panel = pd.concat([
    mcs_2022[cols], 
    mcs_2023[cols],
    mcs_2024[cols],
    mcs_2025[cols]])

In [57]:
ses_demographic_panel[['hh_size_numeric', 'hh_size_cat']] = ses_demographic_panel['hh_size'].apply(lambda x: pd.Series(ses.clean_hh_size(x)))
ses_demographic_panel['kids_clean_count'] = ses_demographic_panel.apply(ses.clean_children_variable, axis=1)
ses_demographic_panel['income_clean'] = ses_demographic_panel['hh_income'].apply(ses.clean_income_variable)
ses_demographic_panel['education_clean'] = ses_demographic_panel['education'].apply(ses.clean_education)
ses_demographic_panel['age_clean'] = ses_demographic_panel['age'].apply(ses.clean_age)
ses_demographic_panel['gender_clean'] = ses_demographic_panel['gender'].apply(ses.clean_gender_variable)
ses_demographic_panel['race_clean'] = ses_demographic_panel.apply(ses.clean_race_ethnicity, axis=1)
# ses_demographic_panel['disability_clean'] = ses_demographic_panel['disability'].apply(ses.clean_disability)
ses_demographic_panel['work_status_clean'] = ses_demographic_panel['work_status'].apply(ses.clean_work_status)
ses_demographic_panel['hh_vehicles_clean'] = ses_demographic_panel['hh_vehicles'].apply(ses.clean_vehicles)

In [62]:
model_df = stated_telework.merge(ses_demographic_panel, how = 'inner', on = 'record_id')

In [64]:
model_df['gender_clean'] = model_df['gender_clean'].fillna('Unknown')
model_df['race_clean'] = model_df['race_clean'].fillna('Unknown')

In [65]:
full_locations_df = pd.read_csv("data/processed/processed/full_google_trip_details_.csv")
cols = ['record_id','SurveyYear']
unique_ids = pd.concat([mcs_2022[cols], 
                        mcs_2023[cols], 
                        mcs_2024[cols], 
                        mcs_2025[cols]])
full_locations_df = full_locations_df.merge(unique_ids, how = 'inner', on = 'record_id')

In [66]:
mask = full_locations_df['home_location_lat'].notna() & full_locations_df['home_location_lon'].notna()
coords_to_search = list(zip(full_locations_df.loc[mask, 'home_location_lat'], 
                            full_locations_df.loc[mask, 'home_location_lon']))
if len(coords_to_search) > 0:
    results = rg.search(coords_to_search)
    full_locations_df.loc[mask, 'home_state'] = [x['admin1'] for x in results]
    full_locations_df.loc[mask, 'home_county'] = [x['admin2'] for x in results]
    full_locations_df.loc[mask, 'home_city'] = [x['name'] for x in results]
    
mask = full_locations_df['work_location_lat'].notna() & full_locations_df['work_location_lon'].notna()
coords_to_search = list(zip(full_locations_df.loc[mask, 'work_location_lat'], 
                            full_locations_df.loc[mask, 'work_location_lon']))
if len(coords_to_search) > 0:
    results = rg.search(coords_to_search)
    full_locations_df.loc[mask, 'work_state'] = [x['admin1'] for x in results]
    full_locations_df.loc[mask, 'work_county'] = [x['admin2'] for x in results]
    full_locations_df.loc[mask, 'work_city'] = [x['name'] for x in results]

Loading formatted geocoded file...


In [67]:
geo_cols = ['home_state', 'home_county', 'home_city', 'work_state', 'work_county', 'work_city']
full_locations_df[geo_cols] = full_locations_df[geo_cols].replace('nan', np.nan)
full_locations_df[geo_cols] = full_locations_df[geo_cols].replace('', 'District of Columbia')

In [68]:
full_locations_df["non_regional_work_states"] = full_locations_df["work_state"].isin([
    "California", "Colorado", "Florida", "Georgia", "Illinois", "Kansas", "Maine",
    "Massachusetts", "New Jersey", "New York", "North Carolina", "Tennessee", "Texas",
    "Washington"
]).astype(int)

In [69]:
cbsa_data = pd.read_csv("data/processed/processed/cbsa_data.csv")

In [70]:
full_locations_df.loc[full_locations_df.work_state == 'Washington, D.C.', 'work_state'] = 'District of Columbia'
full_locations_df.loc[full_locations_df.home_state == 'Washington, D.C.', 'home_state'] = 'District of Columbia'
full_locations_df['work_county'] = full_locations_df['work_county'].str.replace(r'^City of\s+(.*)', r'\1 city', regex=True)
full_locations_df['home_county'] = full_locations_df['home_county'].str.replace(r'^City of\s+(.*)', r'\1 city', regex=True)

In [71]:
full_locations_df_ = full_locations_df.merge(cbsa_data[[
    'CBSA Code',
    'CBSA Title',
    'County/County Equivalent',
    'State Name',
    'Central/Outlying County'
]], 
 how = 'left', 
 left_on = ['home_county', 'home_state'], 
 right_on = ['County/County Equivalent','State Name'] )

In [72]:
cols_to_rename = [
    'CBSA Code',
    'CBSA Title',
    'County/County Equivalent',
    'State Name',
    'Central/Outlying County'
]

In [73]:
rename_map = {
    col: f"home_{col.lower().replace(' ', '_').replace('/', '_')}"
    for col in cols_to_rename
}

In [74]:
full_locations_df_ = full_locations_df_.rename(columns=rename_map)

In [75]:
full_locations_df_ = full_locations_df_.merge(cbsa_data[[
    'CBSA Code',
    'CBSA Title',
    'County/County Equivalent',
    'State Name',
    'Central/Outlying County'
]], 
  how = 'left', 
  left_on = ['work_county', 'work_state'], 
  right_on = ['County/County Equivalent','State Name'] )

In [76]:
rename_map = {
    col: f"work_{col.lower().replace(' ', '_').replace('/', '_')}"
    for col in cols_to_rename
}
full_locations_df_ = full_locations_df_.rename(columns=rename_map)

In [77]:
full_locations_df_.loc[full_locations_df_.home_cbsa_code.isna(), 'home_central_outlying_county'] = 'Outside_CBSA_or_Unclassified'

In [78]:
full_locations_df_.loc[full_locations_df_.home_cbsa_code.isna(), 'home_cbsa_title'] = 'Outside_CBSA_or_Unclassified'

In [79]:
full_locations_df_["core_cbsa"] = full_locations_df_.home_cbsa_title.isin(['Washington-Arlington-Alexandria, DC-VA-MD-WV']).astype(int)

In [80]:
full_locations_df_["expanded_cbsa"] = full_locations_df_.home_cbsa_title.isin(["Washington-Arlington-Alexandria, DC-VA-MD-WV", "Baltimore-Columbia-Towson, MD"]).astype(int)

In [81]:
full_locations_df_["fringe_counties"] = full_locations_df_.home_cbsa_title.isin([
    "Saint Mary's County","Caroline County",
    "Kent County","Garrett County",
    "Westmoreland County","Northumberland County"
]).astype(int)

In [82]:
model_df = model_df.merge(full_locations_df_, on = 'record_id', how = 'inner')

In [89]:
cols = ['home_state',
       'home_county',
        #'home_city', 
        'home_cbsa_code', 'home_cbsa_title',
       'home_county_county_equivalent', 'home_state_name',
       'home_central_outlying_county', ]

In [93]:
county_employed_tbl = p1_wg.county_employed_population(year=2024)


Building county employed population table
ACS 5-Year ending: 2024  (covers 2020-2024)

Fetching B03002 (population by race/ethnicity) for 2024...
  Counties retrieved: 24
Fetching B01001 variants (age by race) for 2024...
Fetching C23002 variants (employment by race) for 2024...

  Total estimated employed population: 3,187,330
  Counties: 24
  Columns: ['non-hispanic white', 'non-hispanic black', 'hispanic', 'other']
  Age bins: ['18-34', '35-54', '55+']


In [95]:
county_employed_tbl.index = county_employed_tbl.index.map(p1_wg.clean_md_county_name)

In [98]:
model_df["home_county"] = model_df["home_county"].map(p1_wg.clean_md_county_name)

In [334]:
model_weights = p1_wg.compute_md_poststrat_weights(model_df, county_employed_tbl)

/Users/alibishokputov/tprg-commuter-survey/paper_1_weighting.py:126: FutureWarning: The previous implementation of stack is deprecated and will be removed in a future version of pandas. See the What's New notes for pandas 2.1.0 for details. Specify future_stack=True to adopt the new implementation and silence this warning.
  pop_geo.stack(level=[0, 1])
/Users/alibishokputov/tprg-commuter-survey/paper_1_weighting.py:236: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df["w_used"] = df["w_raw"].fillna(1.0)


In [336]:
mcs_2025['mobile_2025'] = ((mcs_2025['outside_home_work_loc_type_in_person'].isin(['variable_job_sites', 'drive_travel_between_job_sites'])) | (mcs_2025['outside_home_work_loc_type_hybrid'].isin(['variable_job_sites', 'drive_travel_between_job_sites']))).astype(int)
model_weights = model_weights.merge(mcs_2025[['record_id','mobile_2025']], on = 'record_id', how = 'left')
model_weights['work_remote_clean_mobile'] = model_weights['work_remote_clean']
model_weights.loc[model_weights.mobile_2025 == 1, 'work_remote_clean_mobile'] ='Mobile'
model_weights.loc[model_weights.mobile_2025 == 1, 'work_remote'] ='multiple_worksites'

In [1223]:
mcs_2025_chester = pd.read_csv('data/mcs_2025_main_table.csv')

In [ ]:
mcs_2025_chester['work_days_of_week_count'] = mcs_2025_chester['work_days_count']

In [221]:
mcs_2025_chester['commute_days_of_week_count'] = mcs_2025_chester[[
 'commute_days_bike',
 'commute_days_bus',
 'commute_days_carpool',
 'commute_days_drive_alone',
 'commute_days_light_rail',
 'commute_days_metro',
 'commute_days_other',
 'commute_days_ridehail',
]].sum(axis=1)

In [247]:
category_order = ['remote', 'frequent_hybrid', 'infrequent_hybrid', 'in_person']
mcs_2025_chester['work_remote'] = pd.Categorical(
    mcs_2025_chester['work_remote'],
    categories=category_order,
    ordered=True
)

In [228]:
mcs_2022['commute_days_of_week_count'] = mcs_2022[[
 'commute_days_of_week_friday',
 'commute_days_of_week_monday',
 'commute_days_of_week_saturday',
 'commute_days_of_week_sunday',
 'commute_days_of_week_thursday',
 'commute_days_of_week_tuesday',
 'commute_days_of_week_wednesday',
]].sum(axis=1)

In [233]:
category_order = ['<1', '1', '2', '3','4', '5+']
mcs_2022['commute_days_per_week'] = pd.Categorical(
    mcs_2022['commute_days_per_week'],
    categories=category_order,
    ordered=True
)

In [236]:
category_order = ['always', 'almost always', 'sometimes', 'never']
mcs_2022['work_remote'] = pd.Categorical(
    mcs_2022['work_remote'],
    categories=category_order,
    ordered=True
)

In [251]:
mcs_2023['difference_work_commute_days'] = mcs_2023.work_days_of_week_count - mcs_2023.commute_days_of_week_count

In [263]:
subset = (mcs_2023.work_days_of_week_count.notna()) & (mcs_2023.work_days_of_week_count != 0)

In [241]:
mcs_2024['commute_days_of_week_count'] = mcs_2024[[
'commute_days_bike',
 'commute_days_bus',
 'commute_days_carpool',
 'commute_days_drive_alone',
 'commute_days_light_rail',
 'commute_days_metro',
 'commute_days_other',
 'commute_days_ridehail',
]].sum(axis=1)

In [245]:
category_order = ['remote', 'hybrid', 'in_person', 'multiple_worksites']
mcs_2024['work_remote'] = pd.Categorical(
    mcs_2024['work_remote'],
    categories=category_order,
    ordered=True
)

In [252]:
mcs_2024['difference_work_commute_days'] = mcs_2024.work_days_of_week_count - mcs_2024.commute_days_of_week_count

In [265]:
subset = (mcs_2024.work_days_of_week_count.notna()) & (mcs_2024.work_days_of_week_count != 0)

In [253]:
mcs_2025_chester['difference_work_commute_days'] = mcs_2025_chester.work_days_of_week_count - mcs_2025_chester.commute_days_of_week_count

In [271]:
subset = (mcs_2025_chester.work_days_of_week_count.notna()) & (mcs_2025_chester.work_days_of_week_count != 0)

In [289]:
model_weights = model_weights.merge(mcs_2022[['record_id', 'commute_days_per_week']], on = 'record_id', how = 'left')

In [339]:
cols = ['record_id', 'commute_days_of_week_count']

In [340]:
commute_data = pd.concat([mcs_2022[cols], 
                         mcs_2023[cols],
                         mcs_2024[cols],
                         mcs_2025_chester[cols]])

In [341]:
model_weights = model_weights.merge(commute_data, on = 'record_id', how = 'left')

In [342]:
cols = ['record_id', 'work_days_of_week_count']

In [343]:
commute_data = pd.concat([
                         mcs_2023[cols],
                         mcs_2024[cols],
                         mcs_2025_chester[cols]])

In [344]:
model_weights = model_weights.merge(commute_data, on = 'record_id', how = 'left')

In [346]:
model_weights.loc[model_weights.work_remote.isin(['always', 'remote']), 'commute_days_of_week_count'] = 0

In [349]:
stated_map = {
    'always':              'Remote',
    'almost always':       'Remote',      
    'sometimes':           'Hybrid',
    'never':               'In-Person',
    # 2023 categories
    'hybrid':              'Hybrid',
    'in_person':           'In-Person',
    # 2024 categories
    'remote':              'Remote',
    # 2025 categories
    'frequent_hybrid':     'Hybrid',
    'infrequent_hybrid':   'Hybrid',
    'fully_remote':        'Remote',
    # All years
    'multiple_worksites':  'Mobile',
}

In [350]:
model_weights['work_remote_final'] = model_weights.work_remote.map(stated_map)

In [376]:
def day_mode(df):
    days = ['sunday', 'monday', 'tuesday', 'wednesday', 'thursday', 'friday', 'saturday']
    to_work_cols = [f'{day}_to_work_time' for day in days]
    from_work_cols = [f'{day}_from_work_time' for day in days]
    
    df['commute_days_count'] = df[to_work_cols].notna().sum(axis=1)
    
    df['am_peak_days'] = 0
    for day in days:
        col = f'{day}_to_work_time'
        df['am_peak_days'] += (df[col] == 'morning_06_10').astype(int)

    df['am_peak_share'] = np.where(
        df['commute_days_count'] > 0,
        df['am_peak_days'] / df['commute_days_count'],
        np.nan
    )
    # --- PM peak (conservative): afternoon_14_18 departure from work ---
    df['pm_peak_days'] = 0
    for day in days:
        col = f'{day}_from_work_time'
        df['pm_peak_days'] += (df[col] == 'afternoon_14_18').astype(int)

    df['pm_peak_share'] = np.where(
        df['commute_days_count'] > 0,
        df['pm_peak_days'] / df['commute_days_count'],
        np.nan
    )

    df['any_peak_days'] = 0
    for day in days:
        to_col = f'{day}_to_work_time'
        from_col = f'{day}_from_work_time'
        df['any_peak_days'] += (
            (df[to_col] == 'morning_06_10') | 
            (df[from_col] == 'afternoon_14_18')
        ).astype(int)

    df['any_peak_share'] = np.where(
        df['commute_days_count'] > 0,
        df['any_peak_days'] / df['commute_days_count'],
        np.nan
    )

    # --- Robustness: broader PM peak including evening ---
    df['pm_peak_broad_days'] = 0
    for day in days:
        col = f'{day}_from_work_time'
        df['pm_peak_broad_days'] += (
            df[col].isin(['afternoon_14_18', 'evening_18_21'])
        ).astype(int)

    df['pm_peak_broad_share'] = np.where(
        df['commute_days_count'] > 0,
        df['pm_peak_broad_days'] / df['commute_days_count'],
        np.nan
    )
    return df[[
        'record_id',
        'commute_days_count',
        'am_peak_days',
        'am_peak_share',
        'pm_peak_days',
        'pm_peak_share',
        'any_peak_days',
        'any_peak_share',
        'pm_peak_broad_days',
        'pm_peak_broad_share'
    ]]

In [377]:
df_2023 = day_mode(mcs_2023)

In [379]:
cols = [
     'commute_days_count',
        'am_peak_days',
        'am_peak_share',
        'pm_peak_days',
        'pm_peak_share',
        'any_peak_days',
        'any_peak_share',
        'pm_peak_broad_days',
        'pm_peak_broad_share'
]

In [382]:
df_2024 = day_mode(mcs_2024)

In [385]:
df_2025 = day_mode(mcs_2025_chester)

In [417]:
def mode_shares(df):
    auto_modes = ['commute_days_drive_alone', 'commute_days_carpool', 'commute_days_ridehail']
    transit_modes = ['commute_days_bus', 'commute_days_train', 'commute_days_metro', 'commute_days_heavy_rail', 'commute_days_light_rail']
    active_modes = ['commute_days_walk', 'commute_days_bike', 'commute_days_electric_scooter']
    mode_cols = {
        'Drive alone': ['commute_days_drive_alone'],
        'Shared ride': ['commute_days_carpool', 'commute_days_ridehail'],
        'Transit': ['commute_days_bus', 'commute_days_train', 'commute_days_metro', 'commute_days_heavy_rail', 'commute_days_light_rail'],
        'Active': ['commute_days_walk', 'commute_days_bike', 'commute_days_electric_scooter'],
    }
    for category, cols in mode_cols.items():
        existing_cols = [c for c in cols if c in df.columns]
        df[f'days_{category}'] = df[existing_cols].fillna(0).sum(axis=1)
    mode_category_cols = [f'days_{cat}' for cat in mode_cols.keys()]
    df['primary_mode_collapsed'] = df[mode_category_cols].idxmax(axis=1)
    df['primary_mode_collapsed'] = df['primary_mode_collapsed'].str.replace('days_', '')
    df.loc[df[mode_category_cols].sum(axis=1) == 0, 'primary_mode_collapsed'] = np.nan
    return df[['record_id','primary_mode_collapsed']]

In [418]:
modes_2023 = mode_shares(mcs_2023)

In [419]:
modes_2024 = mode_shares(mcs_2024)

In [420]:
modes_2025 = mode_shares(mcs_2025_chester)

In [426]:
modes = pd.concat([modes_2023, modes_2024, modes_2025])

In [430]:
commute_time = pd.concat([df_2023, df_2024, df_2025])

In [431]:
model_weights = model_weights.merge(commute_time, how = 'left', on='record_id')

In [429]:
model_weights = model_weights.merge(modes, how = 'left', on='record_id')

In [439]:
model_weights['SurveyYear'] = model_weights['SurveyYear'].astype(int)

In [442]:
df_subset = model_weights[model_weights.SurveyYear >= 2023]

In [456]:
df_subset['hh_size_clean_count'] = df_subset['hh_size']
df_subset.loc[df_subset.hh_size > 15, 'hh_size_clean_count'] = np.nan
df_subset['hh_size_clean_count'].value_counts(dropna=False)

/var/folders/zq/8tgyr_zs08gc273_0pj2rs440000gn/T/ipykernel_46489/3029491652.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_subset['hh_size_clean_count'] = df_subset['hh_size']


hh_size_clean_count
2.0     863
1.0     661
3.0     595
4.0     468
5.0     190
6.0      93
7.0      30
8.0      13
NaN       3
10.0      3
9.0       2
11.0      1
12.0      1
Name: count, dtype: int64

In [457]:
df_subset['household_lifecycle'] = 'Adults-Shared'
df_subset.loc[df_subset.hh_size_clean_count == 1, 'household_lifecycle'] = 'Solitary'
df_subset.loc[(df_subset.hh_size_clean_count > 1) & (df_subset.kids_clean_count > 0), 'household_lifecycle'] = 'Family'

/var/folders/zq/8tgyr_zs08gc273_0pj2rs440000gn/T/ipykernel_46489/39408479.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_subset['household_lifecycle'] = 'Adults-Shared'


In [467]:
SOC_teleworkability = pd.read_csv('data/processed/processed/final_SOC_teleworkable_dataset_Jan7.csv')

In [469]:
def norm(s: Optional[str]) -> str:
    if s is None or (isinstance(s, float) and pd.isna(s)):
        return ""
    s = str(s).strip().lower()
    s = re.sub(r"[^\w\s]", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

In [470]:
SOC_teleworkability['work_industry_text_raw'] = SOC_teleworkability['work_industry_text_raw'].apply(norm)

In [471]:
exact_map: Dict[str, str] = {
        "wood working": "Manufacturing",
        "restaurant": "Accommodation and Food Services",
        "telecommunications": "Information",
        "party planning": "Administrative and Support and Waste Management and Remediation Services",
        "professional mobile dj services": "Arts Entertainment and Recreation",
        "childcare": "Health Care and Social Assistance",
        "therapy": "Health Care and Social Assistance",
        "truck driver": "Transportation and Warehousing",
        "cleaning": "Administrative and Support and Waste Management and Remediation Services",
        "delivery": "Transportation and Warehousing",
        "delivery services": "Transportation and Warehousing",
        "commercial design": "Professional Scientific and Technical Services",
        "design primarily commercial design": "Professional Scientific and Technical Services",
        "religion": "Other Services (except Public Administration)",
        "spiritual": "Other Services (except Public Administration)",
        "music": "Arts Entertainment and Recreation",
        "web": "Professional Scientific and Technical Services",
        "consulting": "Professional Scientific and Technical Services",
        "environmental protection": "Professional Scientific and Technical Services",
        "administrative assistant": "Administrative and Support and Waste Management and Remediation Services",
        "customer service": "Administrative and Support and Waste Management and Remediation Services",
        "interior residential organizing": "Administrative and Support and Waste Management and Remediation Services",
    }

In [472]:
SOC_teleworkability['naics_industry_final'] = SOC_teleworkability['naics_industry_name']

In [473]:
for key, val in exact_map.items():
    mask = (
        (SOC_teleworkability["naics_industry_final"] == "Other") &
        (SOC_teleworkability["work_industry_text_raw"] == key)
    )
    SOC_teleworkability.loc[mask, "naics_industry_final"] = val

In [474]:
SOC_teleworkability.loc[SOC_teleworkability.naics_industry_final == 'Other', 'naics_industry_final'] = np.nan

In [475]:
exact_map = {
    "animal caretaker": "Other Services (except Public Administration)",  
    "hair stylist": "Other Services (except Public Administration)",      
    "recovery coach": "Health Care and Social Assistance",                
    "soup kitchen": "Health Care and Social Assistance",                 
    "window cleaned": "Administrative and Support and Waste Management and Remediation Services",  
}

In [476]:
for key, val in exact_map.items():
    mask = (
        (SOC_teleworkability["naics_industry_final"].isna()) &
        (SOC_teleworkability["work_title"] == key)
    )
    SOC_teleworkability.loc[mask, "naics_industry_final"] = val

In [1298]:
SOC_teleworkability.columns

Index(['Unnamed: 0', 'record_id', 'work_title', 'onetsoccode', 'SOC_Title_V2',
       'SOC_Title_V1', 'SOC_Title_V1_', 'Non_Matches_V2_V1', 'Match Type',
       'teleworkable', 'Detailed Match Notes', 'clean_work_title',
       'naics_industry_name', 'work_industry', 'SOC_Code', 'Teleworkable_Flag',
       'Short_SOC_Title', 'Short_SOC_Code', 'SurveyYear', 'SOC_Title_V3',
       'onetsoccode_V3', 'teleworkable_V3', 'General Match Type',
       'Detailed Match Note (Rationale)', 'SOC_Title_Final', 'SOC_Code_Final',
       'SOC_Teleworkable_Final', 'Match_Type_Final', 'work_industry_raw',
       'work_industry_text_raw', 'naics_industry_final'],
      dtype='object')

In [1306]:
cols = [
          'record_id',
           'Match_Type_Final'
]

In [1309]:
SOC_teleworkability[cols].to_csv('data/SOC_data.csv')

In [481]:
df_subset = df_subset.merge(SOC_teleworkability[['record_id', 'SOC_Teleworkable_Final', 'naics_industry_final']], 
                            on = 'record_id',
                            how = 'left')

In [490]:
mcs_2024['reported_commute_time'] = mcs_2024[['commute_time_to_work', 'commute_time_from_work']].mean(axis=1)
mcs_2025_chester['reported_commute_time'] = mcs_2025_chester[['reported_commute_minutes_to_work', 'reported_commute_minutes_from_work']].mean(axis=1)
mcs_2023['reported_commute_time'] = mcs_2023['commute_time']

In [507]:
cols = ['record_id', 'reported_commute_time']
reported_commute_time = pd.concat([mcs_2023[cols],mcs_2024[cols],mcs_2025_chester[cols]])

In [509]:
COMMUTE_MAP = {
    '0_1_min': 0.5, 
    '1_5_min': 3.0, 
    '5_15_min': 10.0,
    '15_30_min': 22.5, 
    '30_45_min': 37.5, 
    '45_60_min': 52.5,
    '60_90_min': 75.0, 
    '90_120_min': 105.0, 
    '120+_min': 150.0  
}

In [531]:
def standardize_reported_commute_minutes_df(
    df: pd.DataFrame,
    commute_col: str,
    record_id_col: str = "record_id",
    commute_map=COMMUTE_MAP,
    upper_hard: float = 240,
    topcode: float = 180,
    keep_original: bool = False,
) -> pd.DataFrame:
    s = df[commute_col]
    raw = s.copy()
    raw_str = raw.astype(str)
    is_range = raw_str.isin(commute_map.keys())
    mapped = raw_str.map(commute_map)
    numeric = pd.to_numeric(raw, errors="coerce")
    minutes = mapped.where(is_range, numeric)
    implausible_high = minutes > upper_hard
    minutes = minutes.where(~implausible_high, np.nan)
    minutes = minutes.where(minutes > 0, np.nan)
    was_topcoded = minutes > topcode
    minutes = minutes.clip(upper=topcode)
    out = pd.DataFrame({
        record_id_col: df[record_id_col].values,  # carry through from parent df
        "reported_commute_time_clean": minutes,
        "reported_time_is_range": is_range.astype("int8"),
        "reported_time_implausible_high": implausible_high.astype("int8"),
        "reported_time_topcoded": was_topcoded.astype("int8"),
    })
    if keep_original:
        out[commute_col] = raw.values
    return out

In [543]:
reported_commute_time_df = standardize_reported_commute_minutes_df(reported_commute_time, commute_col="reported_commute_time", record_id_col="record_id")

In [547]:
df_subset = df_subset.merge(reported_commute_time_df, how = 'left', on ='record_id')

In [500]:
def clean_minutes(s: pd.Series, upper=240):
    s = pd.to_numeric(s, errors="coerce")
    s = s.where(s > 0)              
    s = s.where(s <= upper)        
    return s

In [553]:
def build_commute_potential(df, api_col, rep_col, floor_api=3, topcode=180):
    out = df.copy()
    api = clean_minutes(out[api_col])
    rep = clean_minutes(out[rep_col])
    api = api.where(api.isna() | (api >= floor_api), floor_api)
    source = pd.Series(pd.NA, index=out.index, dtype="object")
    flag_large = pd.Series(0, index=out.index, dtype="int8")
    commute = api.copy()
    source.loc[api.notna()] = "api"
    commute.loc[api.isna() & rep.notna()] = rep.loc[api.isna() & rep.notna()]
    source.loc[api.isna() & rep.notna()] = "reported"
    both = api.notna() & rep.notna()
    override_rep = both & (api < 5) & (rep >= 15)
    commute.loc[override_rep] = rep.loc[override_rep]
    source.loc[override_rep] = "reported_override"
    override_api = both & (rep < 5) & (api >= 15)
    commute.loc[override_api] = api.loc[override_api]
    source.loc[override_api] = "api_override"
    ratio = pd.Series(np.nan, index=out.index)
    ratio.loc[both] = np.maximum(api.loc[both], rep.loc[both]) / np.minimum(api.loc[both], rep.loc[both])
    flag_large.loc[both & (ratio >= 3)] = 1
    commute = commute.clip(upper=topcode)
    out["estimated_one_way_commute_time"] = commute
    out["commute_potential_source"] = source.fillna("missing")
    out["commute_potential_large_discrepancy"] = flag_large
    return out

In [554]:
df_subset_final = build_commute_potential(df_subset, 'Duration', 'reported_commute_time_clean', floor_api=3, topcode=180)

In [580]:
MODE_SPEEDS = {
    'Drive alone':  30,  
    'Shared ride':  27,  
    'Transit':      14,  
    'Active':        6, 
}
DEFAULT_SPEED = 25       

In [581]:
def build_integrated_distance(
    df: pd.DataFrame,
    api_distance_col: str = 'Distance',
    reported_time_col: str = 'reported_commute_time_clean',
    mode_col: str = 'primary_mode_collapsed',
    mode_speed_map: dict = MODE_SPEEDS,
    default_speed: float = DEFAULT_SPEED,
    plausible_min: float = 0.5,
    plausible_max: float = 150.0,
    topcode_distance: float = 150.0,
) -> pd.DataFrame:
    """
    1. API distance where 0.5 ≤ distance ≤ 150 miles
    2. API distance = 0 or < 0.5: convert reported time using mode-specific speed
    3. API distance missing: convert reported time using mode-specific speed
    4. API distance > 150: flag implausible, use reported-time conversion if available
    5. Remaining missing: exclude from distance analyses
    """
    out = df.copy()
    n = len(out)
    
    api_dist = pd.to_numeric(out[api_distance_col], errors='coerce')
    rep_time = pd.to_numeric(out[reported_time_col], errors='coerce')
    
    out['distance_api_raw'] = api_dist
    
    api_flag = pd.Series('missing', index=out.index)
    api_flag[api_dist.notna() & (api_dist >= plausible_min) & (api_dist <= plausible_max)] = 'valid'
    api_flag[api_dist.notna() & (api_dist < plausible_min)] = 'zero_or_low'
    api_flag[api_dist.notna() & (api_dist > plausible_max)] = 'implausible_high'
    out['distance_api_flag'] = api_flag
    
    if mode_col in out.columns:
        mode_speed = out[mode_col].map(mode_speed_map).fillna(default_speed)
    else:
        mode_speed = pd.Series(default_speed, index=out.index)
    
    out['distance_mode_speed'] = mode_speed
    
    time_to_dist = (rep_time / 60) * mode_speed
    
    distance = pd.Series(np.nan, index=out.index)
    source = pd.Series('missing', index=out.index)
    
    mask_api_valid = (api_flag == 'valid')
    distance[mask_api_valid] = api_dist[mask_api_valid]
    source[mask_api_valid] = 'api'
    
    mask_zero = (api_flag == 'zero_or_low')
    mask_zero_has_time = mask_zero & time_to_dist.notna() & (time_to_dist > 0)
    mask_zero_no_time = mask_zero & ~(time_to_dist.notna() & (time_to_dist > 0))
    
    distance[mask_zero_has_time] = time_to_dist[mask_zero_has_time]
    source[mask_zero_has_time] = 'time_converted_zero_api'
    source[mask_zero_no_time] = 'missing_zero_api_no_time'
    
    mask_missing = (api_flag == 'missing')
    mask_missing_has_time = mask_missing & time_to_dist.notna() & (time_to_dist > 0)
    distance[mask_missing_has_time] = time_to_dist[mask_missing_has_time]
    source[mask_missing_has_time] = 'time_converted_missing_api'
    
    mask_high = (api_flag == 'implausible_high')
    mask_high_has_time = mask_high & time_to_dist.notna() & (time_to_dist > 0)
    distance[mask_high_has_time] = time_to_dist[mask_high_has_time]
    source[mask_high_has_time] = 'time_converted_high_api'
    
    was_topcoded = (distance > topcode_distance) & distance.notna()
    distance = distance.clip(upper=topcode_distance)

    out['commute_distance_miles'] = distance
    out['distance_source'] = source
    out['distance_was_topcoded'] = was_topcoded.astype(int)
    
    return out

In [582]:
df = df_subset_final.copy()

In [1366]:
# df = build_integrated_distance(
#     df,
#     api_distance_col='Distance',
#     reported_time_col='reported_commute_time_clean',
#     mode_col='primary_mode_collapsed',  
# )

In [585]:
def validate_distance_vs_time(
    df: pd.DataFrame,
    api_distance_col: str = 'Distance',
    reported_time_col: str = 'reported_commute_time_clean',
    mode_col: str = 'primary_mode_collapsed',
    mode_speed_map: dict = MODE_SPEEDS,
    default_speed: float = DEFAULT_SPEED,
    plausible_min: float = 0.5,
    plausible_max: float = 150.0,
) -> pd.DataFrame:
    """
    For workers with valid API distance and reported time,
    compare the two distance estimates to validate speed assumptions.
    """
    api_dist = pd.to_numeric(df[api_distance_col], errors='coerce')
    rep_time = pd.to_numeric(df[reported_time_col], errors='coerce')
    
    if mode_col in df.columns:
        speed = df[mode_col].map(mode_speed_map).fillna(default_speed)
    else:
        speed = pd.Series(default_speed, index=df.index)
    
    time_dist = (rep_time / 60) * speed
    
    both = (
        api_dist.notna() & (api_dist >= plausible_min) & (api_dist <= plausible_max) &
        time_dist.notna() & (time_dist > 0)
    )
    
    comp = pd.DataFrame({
        'api_distance': api_dist[both],
        'time_distance': time_dist[both],
        'reported_time_min': rep_time[both],
        'mode_speed_mph': speed[both],
        'mode': df[mode_col][both] if mode_col in df.columns else 'unknown',
    })
    return comp

In [1367]:
# comp = validate_distance_vs_time(
#     df,
#     api_distance_col='Distance',
#     reported_time_col='reported_commute_time_clean',
#     mode_col='primary_mode_collapsed',
# )

In [602]:
df = df[~df.gender_clean.isin(["Non_Binary", "Unknown"])] 

In [894]:
df = df[~df.race_clean.isin(["Unknown"])]

In [608]:
df['race_collapsed'] = 'Other/Multiracial'
df.loc[df.race_clean == 'white', 'race_collapsed'] = 'White non-Hispanic'
df.loc[df.race_clean == 'black', 'race_collapsed'] = 'Black non-Hispanic'
df.loc[df.race_clean == 'hispanic_latinx', 'race_collapsed'] = 'Hispanic'

In [611]:
df['age_collapsed'] = '34-54'
df.loc[df.age_clean.isin(['18-24', '25-34']), 'age_collapsed'] = '18-34'
df.loc[df.age_clean.isin(['55-64', '65_plus']), 'age_collapsed'] = '55+'

In [615]:
df['income_collapsed'] = 'Under $50K'
df.loc[df.income_clean.isin(['04_50k_75k', '05_75k_100k']), 'income_collapsed'] = '$50K–$100K'
df.loc[df.income_clean.isin(['06_100k_150k']), 'income_collapsed'] = '$100K–$150K'
df.loc[df.income_clean.isin(['07_Over_150k']), 'income_collapsed'] = 'Over $150K'

In [623]:
cols = ['record_id', 'work_industry', 'work_industry_text']

In [625]:
work_industries = pd.concat([mcs_2023[cols],
                             mcs_2024[cols],
                             mcs_2025[cols]])

In [626]:
df = df.merge(work_industries, on = 'record_id', how='left')

In [634]:
work_industry_to_naics_sector = {
    "agriculture_forestry_fishing": "Agriculture, Forestry, Fishing and Hunting",
    "mining_gas": "Mining, Quarrying, and Oil and Gas Extraction",
    "utilities": "Utilities",
    "construction": "Construction",
    "manufacturing": "Manufacturing",
    "wholesale": "Wholesale Trade",
    "retail": "Retail Trade",
    "transportation_warehousing": "Transportation and Warehousing",
    "information_publishing_media": "Information",
    "banking_finance_insurance": "Finance and Insurance",
    "real_estate": "Real Estate and Rental and Leasing",
    "professional_services": "Professional, Scientific, and Technical Services",
    "admin_support": "Administrative and Support and Waste Management and Remediation Services",
    "education": "Educational Services",
    "healthcare": "Health Care and Social Assistance",
    "arts_entertainment_rec": "Arts, Entertainment, and Recreation",
    "food_lodging": "Accommodation and Food Services",
    "other_services": "Other Services (except Public Administration)",
    "government": "Public Administration",
}

In [635]:
df["naics_sector"] = (df["work_industry"].map(work_industry_to_naics_sector))

In [658]:
df['work_industry_naics_complete'] = df.naics_industry_final.combine_first(df["naics_sector"])

In [660]:
custom_response_to_naics_sector = {
    "other,multiple": 'Nonsense',
    "mitri": 'Nonsense',
    "can you please.": 'Nonsense',
    "childcare": "Health Care and Social Assistance",
    "babysitting": "Health Care and Social Assistance",
    "helping elderly": "Health Care and Social Assistance",
    "telecommunication": "Information",
    "information and communication technology": "Information",
    "biology": "Professional, Scientific, and Technical Services",
    "evaluation and research": "Professional, Scientific, and Technical Services",
    "online surveys": "Professional, Scientific, and Technical Services",
    "law": "Professional, Scientific, and Technical Services",
    "nonprofit": "Other Services (except Public Administration)",
    "non profit": "Other Services (except Public Administration)",
    "religious": "Other Services (except Public Administration)",
    "customer service": "Administrative and Support and Waste Management and Remediation Services",
    "assistant": "Administrative and Support and Waste Management and Remediation Services",
    "operations": "Administrative and Support and Waste Management and Remediation Services",
    "recruiting": "Administrative and Support and Waste Management and Remediation Services",
    "independent contractor": "Administrative and Support and Waste Management and Remediation Services",
    "freelance": "Administrative and Support and Waste Management and Remediation Services",
    "cleaning": "Administrative and Support and Waste Management and Remediation Services",
    "landscaping": "Administrative and Support and Waste Management and Remediation Services",
    "engineering contractor": "Construction",
    "fire/water mitigation": "Construction",
    "private cook": "Accommodation and Food Services",
    "food service": "Accommodation and Food Services",
    "food": "Accommodation and Food Services",
    "chick": "Accommodation and Food Services", 
    "health insurance": "Finance and Insurance",
}

def map_custom_responses_to_naics(series: pd.Series) -> pd.Series:
    """
    Maps custom work-industry responses to NAICS sectors.
    Safely handles NaNs and preserves index alignment.
    """
    out = pd.Series(index=series.index, dtype="object")
    mask = series.notna()
    cleaned = (
        series.loc[mask]
        .astype(str)
        .str.strip()
        .str.lower()
        .str.replace(r"\s+", " ", regex=True)
        .str.replace(r"\s+,", ",", regex=True)
    )
    mapped = cleaned.map(custom_response_to_naics_sector)
    out.loc[mask] = mapped
    return out

In [661]:
df.loc[(df.naics_industry_final.isna()) & (df.work_industry == 'other'), 'custom_inds'] = df.work_industry_text

In [662]:
df['custom_inds'] = map_custom_responses_to_naics(df.work_industry_text)

In [663]:
df['work_industry_naics_complete'] = df.work_industry_naics_complete.combine_first(df["custom_inds"])

In [665]:
df['work_industry_naics_complete'] = df['work_industry_naics_complete'].fillna("Other Services (except Public Administration)")

In [670]:
CANONICAL_NAICS_SECTORS = [
    "Agriculture, Forestry, Fishing and Hunting",
    "Mining, Quarrying, and Oil and Gas Extraction",
    "Utilities",
    "Construction",
    "Manufacturing",
    "Wholesale Trade",
    "Retail Trade",
    "Transportation and Warehousing",
    "Information",
    "Finance and Insurance",
    "Real Estate and Rental and Leasing",
    "Professional, Scientific, and Technical Services",
    "Administrative and Support and Waste Management and Remediation Services",
    "Educational Services",
    "Health Care and Social Assistance",
    "Arts, Entertainment, and Recreation",
    "Accommodation and Food Services",
    "Other Services (except Public Administration)",
    "Public Administration",
]

def _normalize_sector_label(x: object) -> object:
    if pd.isna(x):
        return np.nan
    s = str(x).strip().lower()
    s = re.sub(r"\([^)]*\)", "", s)
    s = s.replace("&", " and ")
    s = re.sub(r"[,/]", " ", s)
    s = re.sub(r"[^a-z0-9\s]", " ", s)   # drop remaining punctuation
    s = re.sub(r"\s+", " ", s).strip()
    return s

_norm_to_canonical = {
    _normalize_sector_label(canon): canon
    for canon in CANONICAL_NAICS_SECTORS
}

def harmonize_naics_sector_strings(series: pd.Series) -> pd.Series:
    norm = series.map(_normalize_sector_label)
    return norm.map(_norm_to_canonical)

df["work_industry_naics_harmonized"] = harmonize_naics_sector_strings(
    df["work_industry_naics_complete"]
)

df["naics_unmatched_flag"] = (
    df["work_industry_naics_complete"].notna()
    & df["work_industry_naics_harmonized"].isna()
).astype("int8")

In [680]:
INDUSTRY_COLLAPSE = {
    'Professional, Scientific, and Technical Services':  'Professional/Finance/Information',
    'Finance and Insurance':                             'Professional/Finance/Information',
    'Information':                                       'Professional/Finance/Information',
    'Real Estate and Rental and Leasing':                'Professional/Finance/Information',
    'Health Care and Social Assistance':                 'Health Care',
    'Educational Services':                              'Education',
    'Retail Trade':                                      'Retail/Food/Hospitality',
    'Accommodation and Food Services':                   'Retail/Food/Hospitality',
    'Arts, Entertainment, and Recreation':               'Retail/Food/Hospitality',
    'Public Administration':                             'Government',
    'Construction':                                      'Construction/Manufacturing/Transport',
    'Transportation and Warehousing':                    'Construction/Manufacturing/Transport',
    'Manufacturing':                                     'Construction/Manufacturing/Transport',
    'Other Services (except Public Administration)':                            'Other Industries',
    'Wholesale Trade':                                                          'Other Industries',
    'Administrative and Support and Waste Management and Remediation Services': 'Other Industries',
    'Agriculture, Forestry, Fishing and Hunting':                               'Other Industries',
    'Utilities':                                                                'Other Industries',
    'Mining, Quarrying, and Oil and Gas Extraction':                            'Other Industries',
}

In [681]:
df["naics_industry_collapsed"] = (df["work_industry_naics_harmonized"].map(INDUSTRY_COLLAPSE))

In [833]:
df = df[df.home_state=='Maryland']

In [837]:
df = df.drop([ 'w_raw','w_in_calibration','w_used','w_trim'],axis=1)

In [839]:
df = p1_wg.compute_md_poststrat_weights(df, county_employed_tbl)

/Users/alibishokputov/tprg-commuter-survey/paper_1_weighting.py:126: FutureWarning: The previous implementation of stack is deprecated and will be removed in a future version of pandas. See the What's New notes for pandas 2.1.0 for details. Specify future_stack=True to adopt the new implementation and silence this warning.
  pop_geo.stack(level=[0, 1])
/Users/alibishokputov/tprg-commuter-survey/paper_1_weighting.py:236: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df["w_used"] = df["w_raw"].fillna(1.0)


In [841]:
p1_wg.weight_diagnostics(df)

,SurveyYear,N,share_in_calibration,sum_w,mean_w,p01,p05,p95,p99,min_w,max_w,ESS
0,2023,962,1.0,947.594131,0.985025,0.457051,0.590049,1.941812,1.980500,0.457051,1.980500,849.679236
1,2024,943,1.0,933.313575,0.989728,0.520373,0.520373,1.912144,3.888727,0.520373,3.888727,738.499303
2,2025,849,1.0,844.574069,0.994787,0.589742,0.589742,2.183744,3.348611,0.589742,3.546889,660.641586


In [842]:
pop_targets = p1_wg.make_pop_targets_from_county_employed_tbl(county_employed_tbl, 
                                                              df, 
                                                              county_col="home_county")

/Users/alibishokputov/tprg-commuter-survey/paper_1_weighting.py:768: FutureWarning: The previous implementation of stack is deprecated and will be removed in a future version of pandas. See the What's New notes for pandas 2.1.0 for details. Specify future_stack=True to adopt the new implementation and silence this warning.
  pop_long = acs.stack([0, 1]).rename("n_pop").reset_index()


In [843]:
cells_full, top_by_weight, top_by_ratio = p1_wg.build_reviewproof_cell_table(df, pop_targets)

In [844]:
cells_full

,SurveyYear,geo_bin,race_bin,age_bin,n_sample,share_sample,n_pop,share_pop,ratio_share_pop_to_sample,weight_mean,weight_min,weight_max
0,2023,Central,hispanic,18-34,43,0.044699,122175.0,0.038473,0.860732,0.860732,0.860732,0.860732
1,2023,Central,hispanic,35-54,22,0.022869,143828.0,0.045292,1.980500,1.980500,1.980500,1.980500
2,2023,Central,hispanic,55+,4,0.004158,71795.0,0.022609,5.437362,1.980500,1.980500,1.980500
3,2023,Central,non-hispanic black,18-34,110,0.114345,241328.0,0.075995,0.664613,0.664613,0.664613,0.664613
4,2023,Central,non-hispanic black,35-54,103,0.107069,279363.0,0.087973,0.821648,0.821648,0.821648,0.821648
5,2023,Central,non-hispanic black,55+,98,0.101871,295748.0,0.093132,0.914218,0.914218,0.914218,0.914218
6,2023,Central,non-hispanic white,18-34,42,0.043659,269217.0,0.084778,1.941812,1.941812,1.941812,1.941812
7,2023,Central,non-hispanic white,35-54,82,0.085239,332214.0,0.104616,1.227321,1.227321,1.227321,1.227321
8,2023,Central,non-hispanic white,55+,175,0.181913,504043.0,0.158725,0.872537,0.872537,0.872537,0.872537
9,2023,Central,other,18-34,53,0.055094,103231.0,0.032508,0.590049,0.590049,0.590049,0.590049


In [845]:
top_by_weight

,SurveyYear,geo_bin,race_bin,age_bin,n_sample,share_sample,n_pop,share_pop,ratio_share_pop_to_sample,weight_mean,weight_min,weight_max
25,2024,Central,hispanic,55+,3,0.003181,71795.0,0.022643,7.117535,3.888727,3.888727,3.888727
24,2024,Central,hispanic,35-54,11,0.011665,143828.0,0.045362,3.888727,3.888727,3.888727,3.888727
67,2025,Outlying,other,18-34,1,0.001178,16596.0,0.005207,4.420629,3.546889,3.546889,3.546889
58,2025,Outlying,hispanic,18-34,1,0.001178,16631.0,0.005218,4.429952,3.546889,3.546889,3.546889
55,2025,Central,other,18-34,7,0.008245,103231.0,0.032388,3.928192,3.546889,3.546889,3.546889
69,2025,Outlying,other,55+,1,0.001178,11765.0,0.003691,3.133809,3.133809,3.133809,3.133809
68,2025,Outlying,other,35-54,2,0.002356,20902.0,0.006558,2.783803,2.783803,2.783803,2.783803
36,2024,Outlying,hispanic,55+,1,0.001060,8447.0,0.002664,2.512229,2.512229,2.512229,2.512229
56,2025,Central,other,35-54,14,0.016490,126577.0,0.039713,2.408282,2.408282,2.408282,2.408282
47,2025,Central,hispanic,35-54,16,0.018846,143828.0,0.045125,2.394441,2.394441,2.394441,2.394441


In [846]:
top_by_ratio

,SurveyYear,geo_bin,race_bin,age_bin,n_sample,share_sample,n_pop,share_pop,ratio_share_pop_to_sample,weight_mean,weight_min,weight_max
25,2024,Central,hispanic,55+,3,0.003181,71795.0,0.022643,7.117535,3.888727,3.888727,3.888727
2,2023,Central,hispanic,55+,4,0.004158,71795.0,0.022609,5.437362,1.980500,1.980500,1.980500
58,2025,Outlying,hispanic,18-34,1,0.001178,16631.0,0.005218,4.429952,3.546889,3.546889,3.546889
67,2025,Outlying,other,18-34,1,0.001178,16596.0,0.005207,4.420629,3.546889,3.546889,3.546889
55,2025,Central,other,18-34,7,0.008245,103231.0,0.032388,3.928192,3.546889,3.546889,3.546889
24,2024,Central,hispanic,35-54,11,0.011665,143828.0,0.045362,3.888727,3.888727,3.888727,3.888727
69,2025,Outlying,other,55+,1,0.001178,11765.0,0.003691,3.133809,3.133809,3.133809,3.133809
68,2025,Outlying,other,35-54,2,0.002356,20902.0,0.006558,2.783803,2.783803,2.783803,2.783803
14,2023,Outlying,hispanic,55+,1,0.001040,8447.0,0.002660,2.558919,1.980500,1.980500,1.980500
36,2024,Outlying,hispanic,55+,1,0.001060,8447.0,0.002664,2.512229,2.512229,2.512229,2.512229


In [848]:
df.to_csv('data/Final_Data_Paper2/final_sample_before_final_cleaning.csv')

In [1000]:
df.work_remote_clean.value_counts(dropna=False)

work_remote_clean
In-Person    1562
Hybrid        556
Remote        343
Mobile        248
NaN            45
Name: count, dtype: int64

In [1001]:
df = df[df.work_remote.notna()]

In [1005]:
df.work_remote_clean.value_counts(dropna=False)

work_remote_clean
In-Person    1562
Hybrid        556
Remote        343
Mobile        248
Name: count, dtype: int64

In [1006]:
df.shape

(2709, 105)

In [1007]:
cols = [
'record_id',
'remote_potential',
'remote_preference',
]

In [1013]:
remote_vars = pd.concat([
    mcs_2023[cols],
    mcs_2024[cols],
    mcs_2025_chester[cols],
])

In [1014]:
df = df.merge(remote_vars, on = 'record_id', how = 'left')

In [1015]:
df.columns

Index(['record_id', 'work_remote', 'work_remote_clean', 'hh_income',
       'education', 'age', 'gender', 'race_ethnicity', 'race_ethnicity_text',
       'disability',
       ...
       'custom_inds', 'work_industry_naics_harmonized', 'naics_unmatched_flag',
       'naics_industry_collapsed', 'w_raw', 'w_in_calibration', 'w_used',
       'w_trim', 'remote_potential', 'remote_preference'],
      dtype='object', length=107)

In [1019]:
df.remote_potential.value_counts(dropna=False)

remote_potential
partial_remote_potential    1205
no_remote_potential          787
full_remote_potential        717
Name: count, dtype: int64

In [1017]:
df.remote_preference.value_counts(dropna=False)

remote_preference
partial_remote_prefer    951
full_remote_prefer       891
no_remote_prefer         867
Name: count, dtype: int64

In [1018]:
df.loc[df.work_remote == 'fully_remote', 'remote_potential'] = 'full_remote_potential'

In [1020]:
cols = ['record_id', 'remote_choice']
remote_choice = pd.concat([mcs_2023[cols], mcs_2024[cols]])

In [1021]:
df = df.merge(remote_choice, on = 'record_id', how = 'left')

In [1136]:
df.shape

(2709, 111)

In [1194]:
cols = ['record_id', 'work_setting', 'work_setting_text']
work_setting = pd.concat([mcs_2023[cols], mcs_2024[cols], mcs_2025_chester[cols]])

In [1195]:
df = df.merge(work_setting, on = 'record_id', how = 'left')

In [1197]:
office_settings = ['office']
non_office_settings = ['retail_store', 'school_university', 'hospital_heathcare',
                       'restaurant_hotel_hospitality', 'factory_warehouse_garage',
                       'construction_site', 'delivery_mobile', 'truck_mobile',
                       'farm_mine', 'theater_stadium_entertainment']

df['office_based'] = df['work_setting'].apply(
    lambda x: 1 if x in office_settings else (0 if x in non_office_settings else pd.NA)
)

In [1198]:
df['office_based'] = (df['work_setting'] == 'office').astype(int)

In [1199]:
df['office_based'].value_counts(dropna=False)

office_based
0    1686
1    1023
Name: count, dtype: int64

In [1200]:
def standardize_category_advanced(val):
    if pd.isna(val):
        return val
    val = str(val).lower().strip()
    val = re.sub(r'[^a-z0-9_ ]+', ' ', val)
    val = val.replace(" ", "_")
    return val  

In [1224]:
dummy_columns = mcs_2025_chester.work_activities.str.get_dummies(sep=',').add_prefix('work_activity_')

In [1225]:
new_cols = [standardize_category_advanced(col) for col in dummy_columns.columns]
dummy_columns.columns = new_cols

In [1227]:
mcs_2025_chester = pd.concat([mcs_2025_chester, dummy_columns], axis=1)

In [1228]:
dummy_columns = mcs_2023.work_activities.str.get_dummies(sep=',').add_prefix('work_activity_')

In [1229]:
new_cols = [standardize_category_advanced(col) for col in dummy_columns.columns]
dummy_columns.columns = new_cols

In [1230]:
mcs_2023 = pd.concat([mcs_2023, dummy_columns], axis=1)

In [1231]:
dummy_columns = mcs_2024.work_activities.str.get_dummies(sep=',').add_prefix('work_activity_')

In [1232]:
new_cols = [standardize_category_advanced(col) for col in dummy_columns.columns]
dummy_columns.columns = new_cols

In [1233]:
mcs_2024 = pd.concat([mcs_2024, dummy_columns], axis=1)

In [1234]:
cols = [ 'record_id', 
         'work_activity_clients',
         'work_activity_colleagues',
         'work_activity_computing',
         'work_activity_driving',
         'work_activity_equipment',
         'work_activity_handling',
         'work_activity_maintenance',
         'work_activity_other',
         'work_activity_paper',
         'work_activity_retail',
         'work_activity_teaching']

In [1235]:
work_activities = pd.concat([
                            mcs_2023[cols],
                           mcs_2024[cols],
                           mcs_2025_chester[cols]])

In [1236]:
df = df.merge(work_activities, on = 'record_id', how = 'left')

In [1238]:
df['desk_task'] = ((df.work_activity_computing==1) | (df.work_activity_paper==1)).astype(int)
df['interpersonal_task'] = ((df.work_activity_colleagues==1) | (df.work_activity_teaching==1) | (df.work_activity_clients==1)).astype(int)
df['physical_task'] = ((df.work_activity_handling==1) | (df.work_activity_driving==1) | (df.work_activity_maintenance==1) | (df.work_activity_equipment==1)).astype(int)

In [1239]:
df.shape

(2709, 125)

In [1241]:
df.work_remote.value_counts(dropna=False)

work_remote
in_person             1079
multiple_worksites     402
never                  400
hybrid                 340
remote                 238
fully_remote           105
frequent_hybrid         94
infrequent_hybrid       51
Name: count, dtype: int64

In [1242]:
df.loc[df.age_collapsed == '34-54', 'age_collapsed'] = '35-54'

In [1243]:
cols = [
'record_id',
'work_setting_aux_self_employed_at_home',
'work_setting_aux_self_employed_outside_home'
]
self_emp = pd.concat([
    mcs_2023[cols],
    mcs_2024[cols]
])
df = df.merge(self_emp, on = 'record_id', how = 'left')

In [1251]:
df.work_setting_aux_self_employed_at_home.value_counts(dropna=False)

work_setting_aux_self_employed_at_home
0.0    1706
NaN     804
1.0     199
Name: count, dtype: int64

In [1254]:
pd.crosstab(df.office_based, df.SOC_Teleworkable_Final, normalize='index').round(2)

SOC_Teleworkable_Final,0.0,1.0
office_based,,
0,0.68,0.32
1,0.25,0.75


In [1263]:
pd.crosstab(df.remote_potential, df.SOC_Teleworkable_Final, normalize='index').round(2)

SOC_Teleworkable_Final,0.0,1.0
remote_potential,,
full_remote_potential,0.32,0.68
no_remote_potential,0.72,0.28
partial_remote_potential,0.48,0.52


In [1264]:
pd.crosstab(df.remote_choice, df.SOC_Teleworkable_Final, normalize='index').round(2)

SOC_Teleworkable_Final,0.0,1.0
remote_choice,,
full_remote_choice,0.37,0.63
no_remote_choice,0.64,0.36
partial_remote_choice,0.36,0.64


In [1278]:
df.loc[df.work_remote_clean == 'Mobile', 'work_remote_clean'] = 'In-Person'

In [1283]:
df['work_remote_clean_1'] = df.work_remote_clean

In [1285]:
df.loc[df.work_remote == 'infrequent_hybrid', 'work_remote_clean_1'] = 'In-Person'

In [1287]:
pd.crosstab(df['work_remote_clean'], df.work_remote_clean_1)

work_remote_clean_1,Hybrid,In-Person,Remote
work_remote_clean,,,
Hybrid,505,51,0
In-Person,0,1810,0
Remote,0,0,343


In [1368]:
def weighted_crosstab(df, row, col, weight, normalize='col', round_digits=1):
    ct = df.groupby([row, col])[weight].sum().unstack(fill_value=0)
    if normalize == 'col':
        pct = ct.div(ct.sum(axis=0), axis=1) * 100
    elif normalize == 'row':
        pct = ct.div(ct.sum(axis=1), axis=0) * 100
    elif normalize == 'all':
        pct = ct / ct.sum().sum() * 100
    result = ct.round(1).astype(str) + ' (' + pct.round(round_digits).astype(str) + '%)'
    result.loc['Total N'] = ct.sum(axis=0).round(1).astype(str)
    return result

In [1316]:
df.shape

(2709, 129)

In [1314]:
df['missing_teleworkable_flag'] = (df.SOC_Teleworkable_Final.isna()).astype(int)

In [1318]:
weighted_crosstab(df, 'missing_teleworkable_flag', 'SurveyYear', 'w_trim', normalize='col')

SurveyYear,2023,2024,2025
missing_teleworkable_flag,,,
0,898.3 (94.8%),896.4 (96.0%),729.2 (91.7%)
1,49.3 (5.2%),36.9 (4.0%),65.7 (8.3%)
Total N,947.6,933.3,795.0


In [1286]:
df.to_csv('data/Final_Data_Paper2/final_sample_cleaned.csv')

In [1249]:
df.shape

(2709, 127)

In [1330]:
remote_reasons_25 = create_dummies_from_text(mcs_2025_chester, 'where_work_factors')

In [1333]:
cols = ['record_id', 
        'remote_like', 
        'remote_like_text', 
        'in_person_like',
        'in_person_like_text']
remote_reasons = pd.concat([mcs_2023[cols], mcs_2024[cols]])
dummy_columns = remote_reasons.remote_like.str.get_dummies(sep=',').add_prefix('remote_like_')
remote_reasons = pd.concat([remote_reasons, dummy_columns], axis=1)
dummy_columns = remote_reasons.in_person_like.str.get_dummies(sep=',').add_prefix('in_person_like_')
remote_reasons = pd.concat([remote_reasons, dummy_columns], axis=1)
remote_reasons_25 = create_dummies_from_text(mcs_2025_chester, 'where_work_factors')
remote_reasons = pd.concat([remote_reasons[['record_id',
                                            'remote_like_do_not_like', 
                                            'remote_like_flexible_schedule', 
                                            'remote_like_more_productive', 
                                            'remote_like_non_work_activities',
                                            'remote_like_not_spend_money_commuting',
                                            'remote_like_not_spend_time_commuting', 
                                            'remote_like_other_reason', 
                                            'remote_like_personalized_work_env', 
                                            'in_person_like_communicate_with_colleagues',
                                            'in_person_like_do_not_like', 
                                            'in_person_like_enjoy_commuting', 
                                            'in_person_like_enjoy_work_env', 
                                            'in_person_like_home_work_boundaries',
                                            'in_person_like_more_productive', 
                                            'in_person_like_other_reason',
                                            'in_person_like_productive_commute',
                                            'in_person_like_prof_relationships']],
                            remote_reasons_25[['record_id', 
                                               'connect_with_colleagues',
                                                 'non_work_activities',
                                                 'other',
                                                 'personalized_work_env',
                                                 'productivity',
                                                 'schedule_flexibility',
                                                 'time_and_cost_of_commuting',
                                                 'work_life_balance']]])

In [1334]:
remote_reasons.shape

(2913, 26)

In [1336]:
def _to_bool_selected(df: pd.DataFrame, col: str) -> pd.Series:
    if col not in df.columns:
        return pd.Series(False, index=df.index)
    s = df[col]
    if pd.api.types.is_bool_dtype(s):
        return s.fillna(False)
    s_num = pd.to_numeric(s, errors="coerce")
    if s_num.notna().any():
        return s_num.fillna(0).astype(float) > 0
    s_str = s.fillna("").astype(str).str.strip().str.lower()
    return s_str.isin({"1", "true", "t", "yes", "y", "selected", "checked"})

def add_workmode_theme_salience(df: pd.DataFrame, prefix: str = "") -> pd.DataFrame:
    out = df.copy()
    themes = {
        f"{prefix}commute_priority": [
            "remote_like_not_spend_time_commuting",
            "remote_like_not_spend_money_commuting",
            "time_and_cost_of_commuting",
        ],
        f"{prefix}flexibility_priority": [
            "remote_like_flexible_schedule",
            "schedule_flexibility",
        ],
        f"{prefix}social_priority": [
            "in_person_like_communicate_with_colleagues",
            "in_person_like_prof_relationships",
            'connect_with_colleagues',
        ],
        f"{prefix}productivity_priority": [
            "remote_like_more_productive",
            "in_person_like_more_productive",
            "productivity",
        ],
        f"{prefix}non_work_activities_priority": [
            "remote_like_non_work_activities",
            'non_work_activities',
        ],
        
        f"{prefix}worklife_priority": [
            "work_life_balance",
            "in_person_like_home_work_boundaries",
        ],
        f"{prefix}workenvironment_priority": [
            "remote_like_personalized_work_env",
            "personalized_work_env",
            'in_person_like_enjoy_work_env'
            
        ],
    }
    for new_col, source_cols in themes.items():
        flags = [_to_bool_selected(out, c) for c in source_cols]
        out[new_col] = np.logical_or.reduce(flags).astype("int8")
    return out

In [1337]:
remote_reasons = add_workmode_theme_salience(remote_reasons)

In [1338]:
remote_reasons.columns

Index(['record_id', 'remote_like_do_not_like', 'remote_like_flexible_schedule',
       'remote_like_more_productive', 'remote_like_non_work_activities',
       'remote_like_not_spend_money_commuting',
       'remote_like_not_spend_time_commuting', 'remote_like_other_reason',
       'remote_like_personalized_work_env',
       'in_person_like_communicate_with_colleagues',
       'in_person_like_do_not_like', 'in_person_like_enjoy_commuting',
       'in_person_like_enjoy_work_env', 'in_person_like_home_work_boundaries',
       'in_person_like_more_productive', 'in_person_like_other_reason',
       'in_person_like_productive_commute',
       'in_person_like_prof_relationships', 'connect_with_colleagues',
       'non_work_activities', 'other', 'personalized_work_env', 'productivity',
       'schedule_flexibility', 'time_and_cost_of_commuting',
       'work_life_balance', 'commute_priority', 'flexibility_priority',
       'social_priority', 'productivity_priority',
       'non_work_activitie

In [1343]:
cols = ['record_id', 'commute_priority', 'flexibility_priority',
       'social_priority', 'productivity_priority',
       'non_work_activities_priority', 'worklife_priority',
       'workenvironment_priority']

In [1342]:
[remote_reasons[col].value_counts(dropna=False) for col in cols]

[commute_priority
 1    1689
 0    1224
 Name: count, dtype: int64,
 flexibility_priority
 1    1460
 0    1453
 Name: count, dtype: int64,
 social_priority
 1    1466
 0    1447
 Name: count, dtype: int64,
 productivity_priority
 0    1481
 1    1432
 Name: count, dtype: int64,
 non_work_activities_priority
 0    2224
 1     689
 Name: count, dtype: int64,
 worklife_priority
 0    1672
 1    1241
 Name: count, dtype: int64,
 workenvironment_priority
 0    1485
 1    1428
 Name: count, dtype: int64]

In [1344]:
df = df.merge(remote_reasons[cols], on = 'record_id', how = 'left')

In [1363]:
weighted_crosstab(df[( df.SOC_Teleworkable_Final==1) & (df.work_remote_clean_1 == "Hybrid")], 
                  'productivity_priority', 'household_lifecycle', 'w_trim', normalize='col')

household_lifecycle,Adults-Shared,Family,Solitary
productivity_priority,,,
0,68.9 (48.3%),50.2 (40.7%),35.6 (43.9%)
1,73.8 (51.7%),73.3 (59.3%),45.5 (56.1%)
Total N,142.7,123.5,81.1


In [1354]:
df.to_csv('data/Final_Data_Paper2/final_sample_cleaned.csv')